# prep_multilingue

Preprocesamiento (no es un experimento: solo genera datos).

Calcula un *embedding* de cada `Description` **en su idioma original** (inglés, malayo, chino...) con el
modelo local `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 dimensiones, mean pooling,
normalizado a largo 1). No traduce y no usa servicios externos. No usa AdoptionSpeed.

Salida: `train.parquet` y `test.parquet` con índice `PetID` y columnas `mle_0..mle_383`
(vacías si la mascota no tiene descripción).

In [1]:
import os, glob, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tok = AutoTokenizer.from_pretrained(MODELO)
mod = AutoModel.from_pretrained(MODELO).eval()

def embeber(textos, lote=64):
    salida = []
    for k in range(0, len(textos), lote):
        enc = tok(textos[k:k + lote], padding=True, truncation=True, max_length=256, return_tensors="pt")
        with torch.no_grad():
            h = mod(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)          # mean pooling (como sentence-transformers)
        salida.append(torch.nn.functional.normalize(v, dim=1).numpy())
    return np.vstack(salida)

C:\Users\eoinb\projects\labo2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5104.43it/s]

In [2]:
RAIZ = "/kaggle/input" if os.path.exists("/kaggle/input") else "../input"
COMP = os.path.dirname(os.path.dirname(glob.glob(f"{RAIZ}/**/train/train.csv", recursive=True)[0]))
SALIDA = "." if os.path.exists("/kaggle") else "../input/lab2-multilingue"
os.makedirs(SALIDA, exist_ok=True)
for conjunto in ["train", "test"]:
    t0 = time.time()
    df = pd.read_csv(f"{COMP}/{conjunto}/{conjunto}.csv", index_col="PetID", usecols=["PetID", "Description"])
    tiene = df["Description"].notna() & (df["Description"].astype(str).str.strip() != "")
    E = embeber(df.loc[tiene, "Description"].astype(str).tolist())
    res = pd.DataFrame(np.nan, index=df.index, columns=[f"mle_{j}" for j in range(E.shape[1])], dtype="float32")
    res.loc[tiene] = E.astype("float32")
    res.to_parquet(os.path.join(SALIDA, f"{conjunto}.parquet"))
    print(conjunto, res.shape, "| con descripción:", int(tiene.sum()), f"| {(time.time() - t0) / 60:.1f} min")

train (14993, 384) | con descripción: 14980 | 13.1 min


test (3972, 384) | con descripción: 3971 | 3.3 min


In [3]:
# Control: frases equivalentes en distintos idiomas deberían quedar cerca (coseno alto)
v = embeber(["very cute kitten, free for adoption", "anak kucing sangat comel, percuma untuk diadopsi",
             "非常可爱的小猫，免费领养", "the car needs new tyres"])
print(np.round(v @ v.T, 2))

[[ 1.    0.93  0.91 -0.04]
 [ 0.93  1.    0.94 -0.01]
 [ 0.91  0.94  1.   -0.01]
 [-0.04 -0.01 -0.01  1.  ]]
